# E-Commerce Analytics System
# Notebook 6: CLI Reporting Tool Development

This notebook develops and tests the reporting logic that will later be used in `report_cli.py`.

It supports:
- Daily / Weekly / Monthly reports
- Date range filtering
- Revenue summary
- Top products
- Previous period comparison
- Input validation
- Graceful error handling


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime,timedelta

DB=Path("database")/"ecommerce.db"

def connect_db():
    try:
        return sqlite3.connect(DB)
    except Exception as e:
        print("Database Connection Error:",e)
        return None


## Report Functions

In [ ]:
def summary_report(conn,start,end):
    query=f'''
    SELECT
        COUNT(DISTINCT o.order_id) total_orders,
        COUNT(DISTINCT o.customer_id) unique_customers,
        ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) revenue
    FROM orders o
    JOIN order_items oi
    ON o.order_id=oi.order_id
    WHERE date(o.order_date)
    BETWEEN date('{start}') AND date('{end}');
    '''
    return pd.read_sql(query,conn)

def top_products(conn,start,end):
    query=f'''
    SELECT
        p.product_name,
        SUM(oi.quantity) quantity,
        ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) revenue
    FROM products p
    JOIN order_items oi
    ON p.product_id=oi.product_id
    JOIN orders o
    ON oi.order_id=o.order_id
    WHERE date(o.order_date)
    BETWEEN date('{start}') AND date('{end}')
    GROUP BY p.product_name
    ORDER BY revenue DESC
    LIMIT 3;
    '''
    return pd.read_sql(query,conn)

def previous_period(start,end):
    s=datetime.fromisoformat(start)
    e=datetime.fromisoformat(end)
    delta=e-s
    prev_end=s-timedelta(days=1)
    prev_start=prev_end-delta
    return prev_start.date(),prev_end.date()


## Input Validation

In [ ]:
def validate_dates(start,end):
    try:
        s=datetime.fromisoformat(start)
        e=datetime.fromisoformat(end)
        if s>e:
            raise ValueError("Start date cannot be after end date.")
        return True
    except Exception as ex:
        print("Invalid Input:",ex)
        return False


## Generate Sample Report

In [ ]:
conn=connect_db()

start='2024-01-01'
end='2024-12-31'

if conn and validate_dates(start,end):
    print("SUMMARY")
    display(summary_report(conn,start,end))

    print("TOP PRODUCTS")
    display(top_products(conn,start,end))

    ps,pe=previous_period(start,end)
    print("Previous Period:",ps,"to",pe)

    print("PREVIOUS SUMMARY")
    display(summary_report(conn,str(ps),str(pe)))


## Edge Case Handling

In [ ]:
tests=[
("Empty Result","2035-01-01","2035-12-31"),
("Single Day","2024-06-01","2024-06-01")
]

for name,s,e in tests:
    print("="*60)
    print(name)
    if validate_dates(s,e):
        try:
            df=summary_report(conn,s,e)
            if df.empty:
                print("No records found.")
            else:
                display(df)
        except Exception as ex:
            print("Handled:",ex)


## Close Connection

In [ ]:
if conn:
    conn.close()
print("CLI development notebook completed.")
